# Dynamic IDK Cascades — Simple Streaming Research Notebook

This notebook implements the Random Forest skip-decision IDK cascade in small, visible steps.

It does **not** require `data/imagenetv2` in this project. Images stream from Hugging Face when dataset/model cells run. Cached model outputs still save to `artifacts/` so expensive inference does not need to be repeated.

In [ ]:
from pathlib import Path
import time

import numpy as np

HF_DATASET = "vaishaal/ImageNetV2"

VARIANT_FOLDERS = {
    "matched-frequency": "imagenetv2-matched-frequency-format-val",
    "threshold-0.7": "imagenetv2-threshold0.7-format-val",
    "top-images": "imagenetv2-top-images-format-val",
}

VARIANT_URLS = {
    "matched-frequency": "https://huggingface.co/datasets/vaishaal/ImageNetV2/resolve/main/imagenetv2-matched-frequency.tar.gz",
    "threshold-0.7": "https://huggingface.co/datasets/vaishaal/ImageNetV2/resolve/main/imagenetv2-threshold0.7.tar.gz",
    "top-images": "https://huggingface.co/datasets/vaishaal/ImageNetV2/resolve/main/imagenetv2-top-images.tar.gz",
}

## 1. Stream ImageNet-V2 from Hugging Face

The Hugging Face dataset streams rows with:

- `jpeg`: PIL image
- `__key__`: archive path, including the class folder
- `__url__`: which ImageNet-V2 archive the row came from

The class label is derived from the numeric folder in `__key__`.

In [ ]:
def label_from_key(key):
    return int(key.split("/")[1])


def stream_imagenet_v2_rows(variant, max_samples=None):
    from datasets import load_dataset

    stream = load_dataset(
        "webdataset",
        data_files={"train": VARIANT_URLS[variant]},
        split="train",
        streaming=True,
    )

    emitted = 0
    for row in stream:
        yield {
            "image": row["jpeg"].convert("RGB"),
            "label": label_from_key(row["__key__"]),
            "key": row["__key__"],
        }
        emitted += 1
        if max_samples is not None and emitted >= max_samples:
            break


def show_stream_examples(variant, n=3):
    for row in stream_imagenet_v2_rows(variant, max_samples=n):
        print(variant, row["label"], row["key"])

### Quick streaming check

Only a 'n' examples from each dataset. It requires internet. Does not store raw images in the project.

In [3]:
for variant in VARIANT_FOLDERS:
    show_stream_examples(variant, n=2)

/Users/abhinavgupta/dynamic-idk-cascades/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


matched-frequency 986 imagenetv2-matched-frequency-format-val/986/838e9b4f82c96029ee35b303222bc3bc208b70a2
matched-frequency 986 imagenetv2-matched-frequency-format-val/986/eb4704a178f0c5f63e9e63cf783630b57b074f11
threshold-0.7 986 imagenetv2-threshold0.7-format-val/986/bbb3cc1e48a859f822615d8ec07b59f45110def1
threshold-0.7 986 imagenetv2-threshold0.7-format-val/986/fbc27c780ef8f6c186398d5ca8e7c486a6e76d5d
top-images 986 imagenetv2-top-images-format-val/986/1568a7ab37ad14b8e1a88e424b3dcd5ed4f3d29a
top-images 986 imagenetv2-top-images-format-val/986/838e9b4f82c96029ee35b303222bc3bc208b70a2


## 2. PyTorch iterable dataset

This wrapper applies the model transform and yields `(image_tensor, label, key)`.

In [4]:
def make_streaming_loader(variant, transform, batch_size=32, max_samples=None, num_workers=0):
    import torch
    from torch.utils.data import DataLoader, IterableDataset

    class ImageNetV2StreamDataset(IterableDataset):
        def __iter__(self):
            for row in stream_imagenet_v2_rows(variant, max_samples=max_samples):
                yield transform(row["image"]), row["label"], row["key"]

    return DataLoader(
        ImageNetV2StreamDataset(),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

## 3. Load pretrained classifiers

This notebook supports ResNet-18, ResNet-34, ResNet-152, and ViT-B/16. The current cascade uses ResNet-152 as the final C-stage model.


In [ ]:
def load_model(model_name, device="cpu"):
    from torchvision import models

    if model_name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT
        model = models.resnet18(weights=weights)
        transform = weights.transforms()
    elif model_name == "resnet34":
        weights = models.ResNet34_Weights.DEFAULT
        model = models.resnet34(weights=weights)
        transform = weights.transforms()
    elif model_name == "resnet152":
        weights = models.ResNet152_Weights.DEFAULT
        model = models.resnet152(weights=weights)
        transform = weights.transforms()
    elif model_name == "vit_base_patch16_224":
        import timm
        from timm.data import create_transform, resolve_data_config

        model = timm.create_model(model_name, pretrained=True)
        transform = create_transform(**resolve_data_config({}, model=model))
    else:
        raise ValueError(f"Unknown model: {model_name}")

    model = model.to(device)
    model.eval()
    return model, transform


## 4. Cache classifier outputs

This is the expensive part. Use `max_samples` for small tests. Set `max_samples=None` for the full 10k-image variant.

In [6]:
def cache_logits(variant, model_name, output_file, batch_size=32, device="cpu", max_samples=None, num_workers=0):
    import torch

    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    model, transform = load_model(model_name, device=device)
    loader = make_streaming_loader(
        variant,
        transform,
        batch_size=batch_size,
        max_samples=max_samples,
        num_workers=num_workers,
    )

    all_probs = []
    all_labels = []
    all_preds = []
    all_times = []
    all_keys = []
    processed = 0

    with torch.inference_mode():
        for images, labels, keys in loader:
            images = images.to(device)
            if device.startswith("cuda"):
                torch.cuda.synchronize()
            start = time.perf_counter()
            logits = model(images)
            if device.startswith("cuda"):
                torch.cuda.synchronize()
            elapsed_ms = (time.perf_counter() - start) * 1000

            probs = torch.softmax(logits, dim=1).cpu().numpy().astype(np.float32)
            all_probs.append(probs)
            all_labels.append(labels.numpy().astype(np.int64))
            all_preds.append(np.argmax(probs, axis=1).astype(np.int64))
            all_times.extend([elapsed_ms / len(images)] * len(images))
            all_keys.extend(list(keys))

            processed += len(images)
            print(f"{model_name} {variant}: processed {processed} images")

    np.savez_compressed(
        output_file,
        probabilities=np.concatenate(all_probs),
        labels=np.concatenate(all_labels),
        predictions=np.concatenate(all_preds),
        times_ms=np.array(all_times, dtype=np.float64),
        keys=np.array(all_keys, dtype=str),
        model_name=np.array(model_name),
        variant=np.array(variant),
    )
    print(f"Saved {output_file}")
    return output_file

### Full cache examples

Run these one at a time. Full CPU runs can take a long time.

In [ ]:
cache_logits("matched-frequency", "resnet18",  "artifacts/matched_resnet18.npz",  batch_size=64)
cache_logits("matched-frequency", "resnet34",  "artifacts/matched_resnet34.npz",  batch_size=32)
cache_logits("matched-frequency", "resnet152", "artifacts/matched_resnet152.npz", batch_size=16)

cache_logits("top-images", "resnet18",  "artifacts/top_resnet18.npz",  batch_size=64)
cache_logits("top-images", "resnet34",  "artifacts/top_resnet34.npz",  batch_size=32)
cache_logits("top-images", "resnet152", "artifacts/top_resnet152.npz", batch_size=16)

cache_logits("threshold-0.7", "resnet18",  "artifacts/threshold07_resnet18.npz",  batch_size=64)
cache_logits("threshold-0.7", "resnet34",  "artifacts/threshold07_resnet34.npz",  batch_size=32)
cache_logits("threshold-0.7", "resnet152", "artifacts/threshold07_resnet152.npz", batch_size=16)


## 5. Random Forest skip model

Features from classifier A:

- confidence
- entropy
- top-1/top-2 margin

Labels:

- `0 = Skip B` when B would IDK
- `1 = Run B` when B would confidently predict

In [58]:
SKIP = 0
PREDICT = 1
CLASSIFICATION_THRESHOLD = 0.9
THRESHOLD_SKIP = 0.3


def load_cache(path):
    return dict(np.load(path, allow_pickle=False))


def confidence(cache):
    return cache["probabilities"].max(axis=1)


def skip_features(probabilities, eps=1e-12):
    conf = probabilities.max(axis=1)
    entropy = -(probabilities * np.log(probabilities + eps)).sum(axis=1)
    top_two = np.partition(probabilities, kth=-2, axis=1)[:, -2:]
    top_two.sort(axis=1)
    margin = top_two[:, 1] - top_two[:, 0]
    return np.column_stack([conf, entropy, margin]).astype(np.float32)


def build_skip_dataset(a_cache, b_cache, a_threshold=None, b_threshold=None):
    a_threshold = CLASSIFICATION_THRESHOLD if a_threshold is None else a_threshold
    b_threshold = CLASSIFICATION_THRESHOLD if b_threshold is None else b_threshold

    a_conf = confidence(a_cache)
    b_conf = confidence(b_cache)
    a_idk = a_conf < a_threshold

    X = skip_features(a_cache["probabilities"][a_idk])
    y = np.where(b_conf[a_idk] < b_threshold, SKIP, PREDICT).astype(np.int64)
    return X, y


def train_rf_skipper(training_pairs, a_threshold=None, b_threshold=None):
    from sklearn.ensemble import RandomForestClassifier

    X_parts = []
    y_parts = []
    for a_path, b_path in training_pairs:
        X, y = build_skip_dataset(load_cache(a_path), load_cache(b_path), a_threshold, b_threshold)
        X_parts.append(X)
        y_parts.append(y)

    X_train = np.concatenate(X_parts)
    y_train = np.concatenate(y_parts)

    rf = RandomForestClassifier(
        n_estimators=50,
        max_depth=4,
        min_samples_leaf=40,
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    )
    rf.fit(X_train, y_train)
    print("training rows:", len(y_train), "skip:", int((y_train == SKIP).sum()), "predict:", int((y_train == PREDICT).sum()))
    return rf


## 6. Cascade evaluation

Compare no-skip, threshold-skip, and RF-skip policies from cached probabilities/timings.

In [59]:
def evaluate_cascade(a_cache, b_cache, c_cache, strategy, rf=None, a_threshold=None, b_threshold=None, threshold_skip=None):
    a_threshold = CLASSIFICATION_THRESHOLD if a_threshold is None else a_threshold
    b_threshold = CLASSIFICATION_THRESHOLD if b_threshold is None else b_threshold
    threshold_skip = THRESHOLD_SKIP if threshold_skip is None else threshold_skip

    labels = a_cache["labels"]
    final_preds = np.empty_like(labels)
    total_time = a_cache["times_ms"].astype(float).copy()

    a_conf = confidence(a_cache)
    b_conf = confidence(b_cache)

    a_successes = 0
    b_successes = 0
    skips = 0
    c_runs = 0
    skip_truth = []
    skip_pred = []

    for i in range(len(labels)):
        if a_conf[i] >= a_threshold:
            final_preds[i] = a_cache["predictions"][i]
            a_successes += 1
            continue

        b_would_idk = b_conf[i] < b_threshold
        should_skip = False

        if strategy == "threshold":
            should_skip = a_conf[i] < threshold_skip
        elif strategy == "rf":
            if rf is None:
                raise ValueError("RF strategy requires a trained rf model")
            features = skip_features(a_cache["probabilities"][i:i+1])
            should_skip = int(rf.predict(features)[0]) == SKIP
        elif strategy != "no-skip":
            raise ValueError("strategy must be no-skip, threshold, or rf")

        if strategy in {"threshold", "rf"}:
            skip_truth.append(SKIP if b_would_idk else PREDICT)
            skip_pred.append(SKIP if should_skip else PREDICT)

        if should_skip:
            skips += 1
            c_runs += 1
            total_time[i] += c_cache["times_ms"][i]
            final_preds[i] = c_cache["predictions"][i]
            continue

        total_time[i] += b_cache["times_ms"][i]
        if b_conf[i] >= b_threshold:
            b_successes += 1
            final_preds[i] = b_cache["predictions"][i]
        else:
            c_runs += 1
            total_time[i] += c_cache["times_ms"][i]
            final_preds[i] = c_cache["predictions"][i]

    result = {
        "strategy": strategy,
        "samples": len(labels),
        "accuracy": float((final_preds == labels).mean()),
        "time_ms_per_image": float(total_time.mean()),
        "a_successes": a_successes,
        "b_successes": b_successes,
        "skips": skips,
        "c_runs": c_runs,
    }

    if skip_truth:
        skip_truth = np.array(skip_truth)
        skip_pred = np.array(skip_pred)
        result["skip_decision_accuracy"] = float((skip_truth == skip_pred).mean())

    return result


## 7. Compare cached research runs

After you cache model outputs, train the skipper and evaluate policies.

In [60]:
from pathlib import Path

training_pairs = [
    ("artifacts/matched_resnet18.npz", "artifacts/matched_resnet34.npz"),
    ("artifacts/top_resnet18.npz", "artifacts/top_resnet34.npz"),
]

test_paths = {
    "a": "artifacts/threshold07_resnet18.npz",
    "b": "artifacts/threshold07_resnet34.npz",
    "c": "artifacts/threshold07_resnet152.npz",
}

required_paths = [path for pair in training_pairs for path in pair] + list(test_paths.values())
missing = [path for path in required_paths if not Path(path).exists()]
if missing:
    print("Missing cached model outputs:")
    for path in missing:
        print(" -", path)
    print("Run the matching cache_logits line before this comparison cell.")
    raise FileNotFoundError("Missing cached model outputs: " + ", ".join(missing))

# Train RF on Matched + Top. Report outcomes only on Threshold 0.7 test data.
# The C stage is the ResNet-152 cache.
rf = train_rf_skipper(training_pairs)

a_test = load_cache(test_paths["a"])
b_test = load_cache(test_paths["b"])
c_test = load_cache(test_paths["c"])

test_dataset_name = "threshold-0.7"
model_labels = ["A: resnet18", "B: resnet34", "C: resnet152"]
model_caches = [a_test, b_test, c_test]

cascade_results = [
    evaluate_cascade(a_test, b_test, c_test, strategy, rf=rf)
    for strategy in ["no-skip", "threshold", "rf"]
]

print("RF training data: matched-frequency + top-images")
print(f"Reported test data: {test_dataset_name}")
print(f"Classification threshold: {CLASSIFICATION_THRESHOLD}")
print(f"Static skip threshold: {THRESHOLD_SKIP}")
print("Final C model: resnet152")


training rows: 12597 skip: 10286 predict: 2311
RF training data: matched-frequency + top-images
Reported test data: threshold-0.7
Classification threshold: 0.9
Static skip threshold: 0.3
Final C model: resnet152


## 8. Printed threshold-0.7 outcomes

The output below is plain text and reports only the threshold-0.7 test data with the shared classification threshold.


In [61]:
print(f"Cascade outcomes on {test_dataset_name} test data")
print(f"Classification threshold: {CLASSIFICATION_THRESHOLD}")
print("Strategy    Samples  Accuracy  Time(ms/img)  A Accepted  B Accepted  Skips  C Runs  Skip Acc")
for result in cascade_results:
    skip_acc = result.get("skip_decision_accuracy")
    skip_acc_text = "n/a" if skip_acc is None else f"{skip_acc:.3f}"
    print(
        f"{result['strategy']:<11} "
        f"{result['samples']:>7}  "
        f"{result['accuracy']:.3f}     "
        f"{result['time_ms_per_image']:>10.2f}  "
        f"{result['a_successes']:>10}  "
        f"{result['b_successes']:>10}  "
        f"{result['skips']:>5}  "
        f"{result['c_runs']:>6}  "
        f"{skip_acc_text:>8}"
    )


Cascade outcomes on threshold-0.7 test data
Classification threshold: 0.9
Strategy    Samples  Accuracy  Time(ms/img)  A Accepted  B Accepted  Skips  C Runs  Skip Acc
no-skip       10000  0.789         203.49        3722        1179      0    5099       n/a
threshold     10000  0.789         200.75        3722        1130   1277    5148     0.376
rf            10000  0.789         202.35        3722         807   3880    5471     0.687


In [62]:
print(f"Routing counts on {test_dataset_name} test data")
print("Strategy    A Accepted  B Accepted  Direct Skips  C Runs")
for result in cascade_results:
    print(
        f"{result['strategy']:<11} "
        f"{result['a_successes']:>10}  "
        f"{result['b_successes']:>10}  "
        f"{result['skips']:>12}  "
        f"{result['c_runs']:>6}"
    )


Routing counts on threshold-0.7 test data
Strategy    A Accepted  B Accepted  Direct Skips  C Runs
no-skip           3722        1179             0    5099
threshold         3722        1130          1277    5148
rf                3722         807          3880    5471


In [63]:
print(f"Standalone model outcomes on {test_dataset_name} test data")
print("Model          Samples  Accuracy  MeanConf  MeanTime(ms)")
for model_name, cache in zip(model_labels, model_caches):
    accuracy = float((cache["predictions"] == cache["labels"]).mean())
    mean_confidence = float(confidence(cache).mean())
    mean_time = float(cache["times_ms"].mean())
    print(
        f"{model_name:<14} "
        f"{len(cache['labels']):>7}  "
        f"{accuracy:.3f}     "
        f"{mean_confidence:.3f}     "
        f"{mean_time:>10.2f}"
    )


Standalone model outcomes on threshold-0.7 test data
Model          Samples  Accuracy  MeanConf  MeanTime(ms)
A: resnet18      10000  0.665     0.695          18.51
B: resnet34      10000  0.700     0.742          33.75
C: resnet152     10000  0.793     0.670         321.19


## 11. Current model skip-decision evaluation

Each skip-decision model predicts whether model B should be skipped on the threshold-0.7 test data with the shared classification threshold. Class 0 means Skip, and class 1 means Predict. The table below compares the Random Forest skipper against the static 0.3 threshold.


In [64]:
def class_metrics(y_true, y_pred, class_id):
    tp = int(((y_true == class_id) & (y_pred == class_id)).sum())
    fp = int(((y_true != class_id) & (y_pred == class_id)).sum())
    fn = int(((y_true == class_id) & (y_pred != class_id)).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return precision, recall, f1


def confusion_counts(y_true, y_pred):
    return np.array([
        [int(((y_true == SKIP) & (y_pred == SKIP)).sum()), int(((y_true == SKIP) & (y_pred == PREDICT)).sum())],
        [int(((y_true == PREDICT) & (y_pred == SKIP)).sum()), int(((y_true == PREDICT) & (y_pred == PREDICT)).sum())],
    ])


a_conf = confidence(a_test)
b_conf = confidence(b_test)
a_idk = a_conf < CLASSIFICATION_THRESHOLD

skip_y_true = np.where(b_conf[a_idk] < CLASSIFICATION_THRESHOLD, SKIP, PREDICT).astype(np.int64)
if len(skip_y_true):
    rf_y_pred = rf.predict(skip_features(a_test["probabilities"][a_idk])).astype(np.int64)
else:
    rf_y_pred = np.array([], dtype=np.int64)
threshold_y_pred = np.where(a_conf[a_idk] < THRESHOLD_SKIP, SKIP, PREDICT).astype(np.int64)

skip_eval_models = [
    ("Random Forest", rf_y_pred),
    (f"Threshold ({THRESHOLD_SKIP})", threshold_y_pred),
]

skip_metric_rows = []
skip_confusions = {}
for model_name, y_pred in skip_eval_models:
    accuracy = float((skip_y_true == y_pred).mean()) if len(skip_y_true) else 0.0
    skip_confusions[model_name] = confusion_counts(skip_y_true, y_pred)
    for class_id, class_name in [(SKIP, "Skip (0)"), (PREDICT, "Predict (1)")]:
        precision, recall, f1 = class_metrics(skip_y_true, y_pred, class_id)
        skip_metric_rows.append({
            "Model": model_name,
            "Class": class_name,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
            "Accuracy": accuracy,
        })

print(f"Skip-decision samples on {test_dataset_name}: {len(skip_y_true)}")
print(f"Classification threshold: {CLASSIFICATION_THRESHOLD}")
print("Model             Class        Precision  Recall  F1-Score  Accuracy")
for row_index, row in enumerate(skip_metric_rows):
    accuracy_text = f"{row['Accuracy']:.2f}" if row_index % 2 == 0 else ""
    print(
        f"{row['Model']:<17} {row['Class']:<12} "
        f"{row['Precision']:.2f}       {row['Recall']:.2f}    {row['F1-Score']:.2f}      {accuracy_text}"
    )

print()
print("Confusion matrices; rows=true class, columns=predicted class")
for model_name, matrix in skip_confusions.items():
    print(model_name)
    print("             Pred Skip  Pred Predict")
    print(f"True Skip    {matrix[0, 0]:>9}  {matrix[0, 1]:>12}")
    print(f"True Predict {matrix[1, 0]:>9}  {matrix[1, 1]:>12}")


Skip-decision samples on threshold-0.7: 6278
Classification threshold: 0.9
Model             Class        Precision  Recall  F1-Score  Accuracy
Random Forest     Skip (0)     0.90       0.69    0.78      0.69
Random Forest     Predict (1)  0.34       0.68    0.45      
Threshold (0.3)   Skip (0)     0.96       0.24    0.39      0.38
Threshold (0.3)   Predict (1)  0.23       0.96    0.37      

Confusion matrices; rows=true class, columns=predicted class
Random Forest
             Pred Skip  Pred Predict
True Skip         3508          1591
True Predict       372           807
Threshold (0.3)
             Pred Skip  Pred Predict
True Skip         1228          3871
True Predict        49          1130
